In [3]:
import datetime
import os
from datetime import datetime, timedelta

import pandas as pd
import requests

url = 'https://api.open-meteo.com/v1/forecast'
data_storage_dir = '/home/coder/tmp/'

current_date = datetime.now()
start_date = current_date - timedelta(days=14)
latitude = 52.02
longitude = 47.47
city = 'Balakovo'

params = {
	"latitude": latitude,
	"longitude": longitude,
	"hourly": ["temperature_2m", "snowfall"],
    "start_date": start_date.strftime('%Y-%m-%d'),
    "end_date": current_date.strftime('%Y-%m-%d')
}

try:
    response = requests.get(url, params=params)
    response.raise_for_status()
except requests.RequestException as e:
    print(f"Ошибка при запросе к сайту: {e}")

data = response.json()

df = pd.DataFrame(
	{
		"datetime":data["hourly"]["time"],
		"temperatue": data["hourly"]["temperature_2m"],
		"snowfall": data["hourly"]["snowfall"],
		"updated_at": current_date + timedelta(hours=3),
		"business_date": f"{start_date.strftime('%Y-%m-%d')} - {current_date.strftime('%Y-%m-%d')}",
	}
)

def assigning_file_name(start_date, end_date, format, city):
	start_date = start_date.strftime('%Y-%m-%d')
	end_date = current_date.strftime('%Y-%m-%d')
	return f'weather_{city}__{start_date}_{end_date}.{format}'

file_name = assigning_file_name(start_date, current_date, 'csv', city)
output_path = os.path.join(data_storage_dir, file_name)
df.to_csv(output_path, sep='~', index=False)

